# Seinfeld English clip extractor

Google Drive의 원본 영상은 복사하지 않습니다. Drive를 마운트한 채 자막에서 표현을 찾고, 선택한 구간만 작은 MP3와 앱용 manifest로 `Seinfeld English Clips` 폴더에 만듭니다.

In [ ]:
!pip -q install pysubs2

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from difflib import SequenceMatcher
import html
import json
import re
import subprocess
import unicodedata
import pysubs2

SOURCE_ROOT = Path('/content/drive/MyDrive/Seinfeld (small size_torrent)')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Seinfeld English Clips')
VIDEO_EXTENSIONS = {'.avi', '.mkv', '.mp4', '.m4v'}
SUBTITLE_EXTENSIONS = {'.srt', '.ass', '.ssa', '.vtt'}

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f'SOURCE_ROOT를 실제 Drive 경로로 수정하세요: {SOURCE_ROOT}')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
videos = sorted(path for path in SOURCE_ROOT.rglob('*') if path.suffix.lower() in VIDEO_EXTENSIONS)
subtitle_files = sorted(path for path in SOURCE_ROOT.rglob('*') if path.suffix.lower() in SUBTITLE_EXTENSIONS)
print(f'영상 {len(videos)}개, 자막 {len(subtitle_files)}개')
print(f'출력 폴더: {OUTPUT_ROOT}')

In [ ]:
def normalize_text(value):
    value = html.unescape(unicodedata.normalize('NFKC', value)).lower()
    value = re.sub(r'<[^>]+>', ' ', value)
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return ' '.join(value.split())

def episode_title(path):
    value = re.sub(r'\.(en|eng)$', '', path.stem, flags=re.IGNORECASE)
    value = re.sub(r'\[[^]]+\]|\([^)]*\)', ' ', value)
    value = re.sub(r'^\s*seinfeld\s*-\s*', '', value, flags=re.IGNORECASE)
    value = re.sub(r'^\s*\d+x\d+\s*-\s*', '', value, flags=re.IGNORECASE)
    value = re.sub(r'^\s*\d{1,3}(?:\s*,\s*\d{1,3})?\s*-\s*', '', value)
    value = re.sub(r'\s*-\s*$', '', value)
    return normalize_text(value)

videos_by_title = {episode_title(video): video for video in videos}

def matching_video(subtitle_path):
    title = episode_title(subtitle_path)
    if title in videos_by_title:
        return videos_by_title[title]
    scored = [(SequenceMatcher(None, title, key).ratio(), video) for key, video in videos_by_title.items()]
    score, video = max(scored, default=(0, None), key=lambda item: item[0])
    return video if score >= 0.82 else None

subtitle_cues = []
for subtitle_path in subtitle_files:
    subtitles = None
    for encoding in ('utf-8', 'cp1252', 'cp949'):
        try:
            subtitles = pysubs2.load(str(subtitle_path), encoding=encoding)
            break
        except (UnicodeDecodeError, LookupError):
            pass
        except Exception:
            break
    if subtitles is None:
        continue
    video = matching_video(subtitle_path)
    for cue in subtitles:
        text = ' '.join(cue.plaintext.splitlines())
        subtitle_cues.append({
            'subtitle': subtitle_path,
            'video': video,
            'start_ms': cue.start,
            'end_ms': cue.end,
            'text': text,
            'normalized': normalize_text(text),
        })

print(f'검색 가능한 자막 줄 {len(subtitle_cues):,}개')

In [ ]:
def format_seconds(value):
    milliseconds = round(value * 1000)
    hours, milliseconds = divmod(milliseconds, 3_600_000)
    minutes, milliseconds = divmod(milliseconds, 60_000)
    seconds, milliseconds = divmod(milliseconds, 1000)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}'

def find_phrase(query, limit=10):
    needle = normalize_text(query)
    matches = [cue for cue in subtitle_cues if needle and needle in cue['normalized']]
    matches = matches[:limit]
    print(f'\n{query!r}: {len(matches)}개 표시')
    for index, match in enumerate(matches):
        video_name = match['video'].name if match['video'] else '영상 자동 연결 안 됨'
        print(f"[{index}] {format_seconds(match['start_ms'] / 1000)} | {video_name}")
        print(f"    {match['text']}")
    return matches

def slugify(value):
    value = unicodedata.normalize('NFKD', value).encode('ascii', 'ignore').decode('ascii').lower()
    return re.sub(r'[^a-z0-9]+', '-', value).strip('-')[:72] or 'clip'

def extract_clip(app_phrase, match, before=3.0, after=3.0, video_path=None):
    video = Path(video_path) if video_path else match.get('video')
    if not video or not video.exists():
        raise FileNotFoundError('영상을 자동 연결하지 못했습니다. video_path에 해당 영상 경로를 지정하세요.')

    start = max(0, match['start_ms'] / 1000 - before)
    end = match['end_ms'] / 1000 + after
    duration = end - start
    if duration <= 0 or duration > 30:
        raise ValueError('클립 길이는 0초보다 길고 30초 이하여야 합니다.')

    output = OUTPUT_ROOT / f'{slugify(app_phrase)}.mp3'
    command = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-ss', f'{start:.3f}', '-i', str(video), '-t', f'{duration:.3f}',
        '-vn', '-ac', '1', '-ar', '44100', '-b:a', '96k', str(output),
    ]
    subprocess.run(command, check=True)

    manifest_path = OUTPUT_ROOT / 'clip-manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    else:
        manifest = {'version': 1, 'clips': {}}
    manifest.setdefault('clips', {})[app_phrase] = {
        'file': output.name,
        'episode': video.name,
        'start': format_seconds(start),
        'end': format_seconds(end),
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'완료: {output.name} ({output.stat().st_size / 1024:.1f} KB)')
    return output

# 1) 검색 결과를 듣고 싶은 장면인지 확인합니다.
matches = find_phrase('No soup for you')

# 2) 첫 결과는 3분 56초의 빵값 실랑이 장면입니다. Run all에서 바로 추출합니다.
no_soup_clip = extract_clip('No soup for you.', matches[0], before=11.0, after=2.0)

## 다른 표현 추출

앱에 표시되는 문구를 `extract_clip`의 첫 번째 인자로 정확히 넣어야 manifest와 연결됩니다. 자막 문구가 다르면 `find_phrase`에는 더 짧은 핵심어를 넣어도 됩니다. 생성된 MP3와 `clip-manifest.json`만 앱의 `audio` 폴더로 가져오면 됩니다.

In [ ]:
# 예시
# matches = find_phrase('serenity now')
# extract_clip('Serenity now.', matches[0], before=2.5, after=3.5)

# matches = find_phrase('yada yada yada')
# extract_clip('Yada, yada, yada.', matches[0], before=3.0, after=3.0)